## Exercise 1

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 5: Backpropagation & Gradientenabstieg
# Niveau: Anfänger
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

# Step-by-step backpropagation for a simple 2-layer network
import numpy as np

def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
def sigmoid_abl(x): s = sigmoid(x); return s*(1-s)

# Netzwerk: 2 Eingaben -> 2 versteckt -> 1 Ausgabe
# Feste Gewichte aus dem klassischen Backprop-Beispiel (Rumelhart et al.)
np.random.seed(42)
W1 = np.array([[0.15, 0.25], [0.20, 0.30]])
b1 = np.array([0.35, 0.35])
W2 = np.array([[0.40, 0.50]])
b2 = np.array([0.60])
lernrate = 0.5

X = np.array([0.05, 0.10])
y_ziel = 0.01

print("=== Backpropagation – Schritt fuer Schritt ===")
print(f"\nEingabe: {X}, Ziel: {y_ziel}")

# ---- VORWÄRTSDURCHLAUF ----
z1 = W1 @ X + b1          # Vorwärtsdurchlauf Schicht 1
a1 = sigmoid(z1)           # Aktivierung Schicht 1
z2 = W2 @ a1 + b2          # Vorwärtsdurchlauf Schicht 2
a2 = sigmoid(z2)           # Aktivierung Ausgabe

print(f"\n--- Vorwaertsdurchlauf ---")
print(f"z1 = W1*X + b1 = {z1.round(6)}")
print(f"a1 = sigmoid(z1) = {a1.round(6)}")
print(f"z2 = W2*a1 + b2 = {z2.round(6)}")
print(f"a2 = sigmoid(z2) = {a2.round(6)}")
verlust = 0.5 * (y_ziel - a2[0])**2
print(f"MSE Verlust = {verlust:.6f}")

# ---- RÜCKWÄRTSDURCHLAUF ----
print(f"\n--- Rueckwaertsdurchlauf ---")

# Gradient an der Ausgabe
d_verlust_a2 = -(y_ziel - a2[0])          # dL/da2
d_a2_z2 = sigmoid_abl(z2[0])              # da2/dz2
delta2 = d_verlust_a2 * d_a2_z2           # dL/dz2
print(f"delta_Ausgabe = {delta2:.6f}")

# Gradient bezüglich W2 und b2
d_verlust_W2 = delta2 * a1
d_verlust_b2 = delta2
print(f"dL/dW2 = {d_verlust_W2.round(6)}")

# Gradient zurück durch versteckte Schicht
delta1 = (W2[0] * delta2) * sigmoid_abl(z1)
d_verlust_W1 = np.outer(delta1, X)
print(f"delta_versteckt = {delta1.round(6)}")
print(f"dL/dW1 =\n{d_verlust_W1.round(6)}")

# ---- GEWICHTS-UPDATE ----
W1_neu = W1 - lernrate * d_verlust_W1
W2_neu = W2 - lernrate * d_verlust_W2.reshape(1,-1)
print(f"\n--- Gewichts-Update (lr={lernrate}) ---")
print(f"W1_neu =\n{W1_neu.round(6)}")
print(f"W2_neu = {W2_neu.round(6)}")

# Neue Vorhersage nach Update
z1_neu = W1_neu @ X + b1; a1_neu = sigmoid(z1_neu)
z2_neu = W2_neu @ a1_neu + b2; a2_neu = sigmoid(z2_neu)
verlust_neu = 0.5 * (y_ziel - a2_neu[0])**2
print(f"\nVerlust vorher: {verlust:.6f}")
print(f"Verlust nachher: {verlust_neu:.6f}")
print(f"Verbesserung: {verlust - verlust_neu:.6f}")


## Exercise 2

**Dataset Used:** Synthetic Moons Data (sklearn.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 5: Backpropagation & Gradientenabstieg
# Niveau: Anfänger
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

# Visualize the effect of different learning rates on training
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42); np.random.seed(42)

X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train); X_test = scaler.transform(X_test)

# Verschiedene Lernraten testen
lernraten = [0.001, 0.01, 0.1, 1.0]
ergebnisse = {}

for lr in lernraten:
    tf.random.set_seed(42)
    modell = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation='relu', input_shape=(2,)),
        tf.keras.layers.Dense(8, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    modell.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=lr),
                   loss='binary_crossentropy', metrics=['accuracy'])
    h = modell.fit(X_train, y_train, epochs=100, verbose=0, validation_data=(X_test, y_test))
    ergebnisse[lr] = h.history
    print(f"LR={lr:.3f}: finaler Verlust={h.history['loss'][-1]:.4f}, Genauigkeit={h.history['val_accuracy'][-1]:.2%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Einfluss der Lernrate auf das Training (SGD)', fontsize=14)

for lr, hist in ergebnisse.items():
    axes[0].plot(hist['loss'], label=f'lr={lr}', linewidth=2)
    axes[1].plot(hist['val_accuracy'], label=f'lr={lr}', linewidth=2)

for ax, titel in zip(axes, ['Trainingsverlust', 'Validierungsgenauigkeit']):
    ax.set_title(titel); ax.set_xlabel('Epoche')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lernrate_einfluss.png', dpi=100)
plt.show()
print("Lernratenvergleich gespeichert: lernrate_einfluss.png")
print("Empfehlung: lr=0.01 ist oft ein guter Startpunkt fuer SGD")


## Exercise 3

**Dataset Used:** Synthetic Circles Data (sklearn.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 5: Backpropagation & Gradientenabstieg
# Niveau: Anfänger
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

# Compare SGD, Adam, RMSprop optimizers
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42); np.random.seed(42)

X, y = make_circles(n_samples=600, noise=0.1, factor=0.4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train); X_test = scaler.transform(X_test)

# Verschiedene Optimierer vergleichen
optimierer = {
    'SGD':          tf.keras.optimizers.SGD(learning_rate=0.1),
    'SGD Momentum': tf.keras.optimizers.SGD(learning_rate=0.1, momentum=0.9),
    'RMSprop':      tf.keras.optimizers.RMSprop(learning_rate=0.01),
    'Adam':         tf.keras.optimizers.Adam(learning_rate=0.01),
}

ergebnisse = {}
for name, opt in optimierer.items():
    tf.random.set_seed(42)
    m = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(2,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    m.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    h = m.fit(X_train, y_train, epochs=100, verbose=0, validation_data=(X_test, y_test))
    ergebnisse[name] = h.history
    print(f"{name:15s}: finale Testgenauigkeit = {h.history['val_accuracy'][-1]:.2%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
stile = ['-', '--', '-.', ':']
for (name, hist), stil in zip(ergebnisse.items(), stile):
    axes[0].plot(hist['loss'],         label=name, linestyle=stil, linewidth=2)
    axes[1].plot(hist['val_accuracy'], label=name, linestyle=stil, linewidth=2)

for ax, t in zip(axes, ['Trainingsverlust', 'Validierungsgenauigkeit']):
    ax.set_title(t); ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Optimierer-Vergleich: SGD vs Adam vs RMSprop', fontsize=13)
plt.tight_layout()
plt.savefig('optimierer_vergleich.png', dpi=100)
plt.show()
print("Optimierervergleich gespeichert: optimierer_vergleich.png")
